# V5W_04 — DHSLP Subject-Independent (5 parole)

Adattato da **EEG_13**. Un solo modello su un set di soggetti TRAIN, early-stop su VAL, valutato su soggetti TEST **mai visti**. Split deterministico (~64/12/24%, seed 42) sui soggetti 5words presenti. **5 classi**, **chance 20%**.

È il test pulito di EEG_38 fatto su dati 5-parole *nativi* (non sottocampionati): il vocabolario piccolo rompe il chance ceiling cross-subject, o no?

**Env: `daniele_311`**, GPU.

## §1 — Config + split subject-independent

In [ ]:
import json, logging, re, random
from collections import defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import balanced_accuracy_score
from tqdm.auto import tqdm
import wandb

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('v5w04')

project_root = next((p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/'.git').exists()), Path.cwd())
FIG_DIR  = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)
CKPT_DIR = project_root / 'models' / 'v5w04'; CKPT_DIR.mkdir(parents=True, exist_ok=True)

N_CHANNELS, N_SAMPLES = 61, 384
N_CLASSES   = 5
DATA_METRIC = 'abs_pcc'
_i2l = json.loads((project_root/'configs'/'label_schemes'/'idx2label_5words.json').read_text())
CLASS_NAMES = [_i2l[str(i)] for i in range(N_CLASSES)]

K_WINDOWS, N_EDGES, D_MODEL, HIDDEN, N_LAYERS, DROPOUT = 8, 16, 64, 128, 2, 0.5
LR, WEIGHT_DECAY, GRAD_CLIP, BATCH_SIZE = 1e-3, 1e-2, 1.0, 32
MAX_EPOCHS, PATIENCE = 200, 40
USE_INSTANCE_NORM, LABEL_SMOOTHING = True, 0.1
USE_AUGMENTATION = True
AUG_NOISE_STD, AUG_AMP_RANGE, AUG_SHIFT_MAX, AUG_MASK_LEN = 0.05, (0.85, 1.15), 15, 20
T_WIN = N_SAMPLES // K_WINDOWS
WANDB_ENTITY, WANDB_PROJECT = 'uras-daniele22-politecnico-di-milano', 'miralis-imagined-speech'

HG_ROOT = project_root / 'data' / '5words_subjects' / 'graphs' / f'hypergraphs_pruned_{DATA_METRIC}'
assert HG_ROOT.exists(), f'Grafi non trovati: {HG_ROOT} — esegui prima V5W_02'

_PAT = re.compile(r'^P(\d+)_S(\d+)$')
ALL_SUBJ = sorted({int(_PAT.match(p.parent.name).group(1))
                   for p in HG_ROOT.rglob('trial_*.pt') if _PAT.match(p.parent.name)})

# Split subject-independent deterministico ~ 64/12/24 %
rng = random.Random(42); shuf = ALL_SUBJ[:]; rng.shuffle(shuf)
n = len(shuf); n_te = max(1, round(n*0.24)); n_va = max(1, round(n*0.12))
SUBJ_TEST  = sorted(shuf[:n_te])
SUBJ_VAL   = sorted(shuf[n_te:n_te+n_va])
SUBJ_TRAIN = sorted(shuf[n_te+n_va:])
log.info(f'Soggetti disponibili: {n}  classi: {CLASS_NAMES}  chance={1/N_CLASSES:.0%}')
log.info(f'TRAIN={len(SUBJ_TRAIN)}  VAL={len(SUBJ_VAL)}  TEST={len(SUBJ_TEST)}')
log.info(f'TEST subj: {SUBJ_TEST}')


## §2 — Dataset (y = parola 0-4)

In [ ]:
def _augment_eeg(x):
    x = x.clone()
    if torch.rand(1) < 0.5: x = x + torch.randn_like(x) * AUG_NOISE_STD
    if torch.rand(1) < 0.5: x = x * torch.empty(1).uniform_(*AUG_AMP_RANGE)
    if torch.rand(1) < 0.5:
        shift = torch.randint(-AUG_SHIFT_MAX, AUG_SHIFT_MAX + 1, (1,)).item(); x = torch.roll(x, shift, dims=1)
    if torch.rand(1) < 0.3:
        T = x.shape[1]; s = torch.randint(0, max(1, T - AUG_MASK_LEN), (1,)).item(); x[:, s:s+AUG_MASK_LEN] = 0.0
    if torch.rand(1) < 0.2:
        ch = torch.randint(0, x.shape[0], (1,)).item(); x[ch] = 0.0
    return x

class EEGRawDataset(Dataset):
    """x (61,384) + y (parola 0-4, diretta). Solo trial _img (V5W_02 globa *_img_*)."""
    def __init__(self, subj_ids, use_instance_norm=True, augment=False):
        self.paths, self.labels = [], []
        self.use_instance_norm, self.augment = use_instance_norm, augment
        subj_ids = set(subj_ids)
        for p in sorted(HG_ROOT.rglob('trial_*.pt')):
            m = _PAT.match(p.parent.name)
            if not m or int(m.group(1)) not in subj_ids: continue
            d = torch.load(p, weights_only=False)
            y = int(d['y'].squeeze()) if isinstance(d['y'], torch.Tensor) else int(d['y'])
            if 0 <= y < N_CLASSES:
                self.paths.append(p); self.labels.append(y)
        log.info(f'  {len(self.paths)} trial caricati ({len(subj_ids)} soggetti)')
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        d = torch.load(self.paths[idx], weights_only=False)
        x = d['x'].float()
        if self.use_instance_norm:
            x = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)
        if self.augment: x = _augment_eeg(x)
        return x, torch.tensor(self.labels[idx], dtype=torch.long)

def make_loaders():
    tr = EEGRawDataset(SUBJ_TRAIN, augment=USE_AUGMENTATION)
    va = EEGRawDataset(SUBJ_VAL,   augment=False)
    te = EEGRawDataset(SUBJ_TEST,  augment=False)
    labels = np.array(tr.labels); counts = np.bincount(labels, minlength=N_CLASSES)
    sample_w = torch.tensor(1.0 / np.clip(counts[labels], 1, None), dtype=torch.float)
    sampler = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)
    kw = dict(num_workers=2, pin_memory=True)
    return (DataLoader(tr, BATCH_SIZE, sampler=sampler, **kw),
            DataLoader(va, BATCH_SIZE, shuffle=False, **kw),
            DataLoader(te, BATCH_SIZE, shuffle=False, **kw))


## §3 — Modello DHSLP

In [ ]:
class HGNNConv(nn.Module):
    def __init__(self, in_ch, out_ch, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_ch, out_ch))
        self.bias = nn.Parameter(torch.zeros(out_ch)) if bias else None
        nn.init.xavier_uniform_(self.weight)
    def forward(self, X, H):
        d_v = H.sum(dim=2).clamp(min=1e-6); d_e = H.sum(dim=1).clamp(min=1e-6)
        Dv = (1.0 / d_v.sqrt()).unsqueeze(-1); De = (1.0 / d_e).unsqueeze(1)
        out = Dv * (X @ self.weight); out = torch.bmm(H.transpose(1, 2), out)
        out = De.transpose(1, 2) * out; out = torch.bmm(H, out); out = Dv * out
        if self.bias is not None: out = out + self.bias
        return out

class DHSLP(nn.Module):
    def __init__(self, n_nodes=N_CHANNELS, T_win=T_WIN, K=K_WINDOWS, n_edges=N_EDGES,
                 d_model=D_MODEL, hidden=HIDDEN, n_classes=N_CLASSES, n_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.K, self.T_win, self.d_model = K, T_win, d_model
        self.E = nn.Parameter(torch.randn(n_edges, d_model) * 0.01)
        self.pos_enc = nn.Parameter(torch.randn(n_nodes, d_model) * 0.01)
        self.node_proj = nn.Sequential(nn.Linear(T_win, d_model), nn.LayerNorm(d_model), nn.ELU())
        dims = [d_model] + [hidden] * n_layers
        self.convs = nn.ModuleList([HGNNConv(dims[i], dims[i+1]) for i in range(n_layers)])
        self.bns = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(n_layers)])
        self.drop = nn.Dropout(dropout); self.clf = nn.Linear(hidden, n_classes)
    def build_dynamic_H(self, feat):
        return torch.softmax(torch.matmul(feat, self.E.T) / (self.d_model ** 0.5), dim=2)
    def forward(self, x):
        B, N, T = x.shape; outs = []
        for k in range(self.K):
            x_k = x[:, :, k*self.T_win:(k+1)*self.T_win]
            feat = self.node_proj(x_k) + self.pos_enc
            H_k = self.build_dynamic_H(feat); out = feat
            for conv, bn in zip(self.convs, self.bns):
                out = conv(out, H_k); out = bn(out.reshape(B*N, -1)).reshape(B, N, -1)
                out = F.relu(out); out = self.drop(out)
            outs.append(out.mean(dim=1))
        return self.clf(torch.stack(outs, dim=1).mean(dim=1))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_m = DHSLP(); log.info(f'device={device}  DHSLP {sum(p.numel() for p in _m.parameters()):,} params'); del _m


## §4 — Train / Eval

In [ ]:
_criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    total_loss, all_labels, all_preds = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x); loss = _criterion(logits, y)
            if train:
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP); optimizer.step()
            total_loss += loss.item() * len(y)
            all_labels.extend(y.cpu().numpy()); all_preds.extend(logits.argmax(1).cpu().numpy())
    return total_loss/len(loader.dataset), balanced_accuracy_score(all_labels, all_preds), np.array(all_labels), np.array(all_preds)

def train_dhslp(run_name, model, tr_l, va_l, te_l, cfg):
    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name=run_name, config=cfg,
                     reinit='finish_previous', settings=wandb.Settings(start_method='thread'))
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS)
    best_val, best_state, pc = 0.0, None, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        tr_loss, tr_b, _, _ = run_epoch(model, tr_l, opt)
        va_loss, va_b, _, _ = run_epoch(model, va_l); sched.step()
        run.log({'train/loss': tr_loss, 'train/bacc': tr_b, 'val/loss': va_loss,
                 'val/bacc': va_b, 'train_val_gap': tr_b - va_b, 'epoch': epoch})
        if va_b > best_val:
            best_val = va_b; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}; pc = 0
        else: pc += 1
        if pc >= PATIENCE: log.info(f'  early stop @ ep {epoch}'); break
    model.load_state_dict(best_state)
    _, te_b, te_lbl, te_pred = run_epoch(model, te_l)
    run.summary['val_bacc'], run.summary['test_bacc'] = best_val, te_b
    try:
        run.log({'confusion_matrix': wandb.plot.confusion_matrix(
            preds=te_pred.tolist(), y_true=te_lbl.tolist(), class_names=CLASS_NAMES)})
    except Exception: pass
    run.finish()
    torch.save({'state_dict': best_state, 'val_bacc': best_val, 'test_bacc': te_b,
                'labels': te_lbl, 'preds': te_pred}, CKPT_DIR / 'dhslp_si.pt')
    log.info(f'  {run_name}: val={best_val:.4f}  test={te_b:.4f}')
    return best_val, te_b, te_lbl, te_pred


## §5 — Run + risultato

In [ ]:
tr_l, va_l, te_l = make_loaders()
log.info(f'Dataset: train={len(tr_l.dataset)} val={len(va_l.dataset)} test={len(te_l.dataset)}')

cfg = dict(notebook='V5W_04', model='DHSLP_SI_5words', n_classes=N_CLASSES,
           k_windows=K_WINDOWS, n_edges=N_EDGES, d_model=D_MODEL, hidden=HIDDEN,
           dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
           patience=PATIENCE, n_train_subj=len(SUBJ_TRAIN), n_test_subj=len(SUBJ_TEST))
val_b, test_b, lbl, pred = train_dhslp('v5w04_DHSLP_SI_5words', DHSLP(), tr_l, va_l, te_l, cfg)

chance = 1 / N_CLASSES
print('='*50)
print('  V5W_04 — DHSLP Subject-Independent 5words')
print('='*50)
print(f'  val bAcc  = {val_b:.4f}')
print(f'  test bAcc = {test_b:.4f}   (chance {chance:.0%})')
print(f'  delta test-chance = {test_b - chance:+.4f}')
print('='*50)
print('  → Se test >> 20%: il vocabolario ridotto rompe il chance ceiling cross-subject.')
print('  → Se test ≈ 20%: il ceiling regge anche a 5 parole native (conferma EEG_38).')
